# Module 4 Community Risk Dashboard

Where the analysis meets people: structures, communities, water supply, and
watersheds downstream of the burn scar.

**This module replaces the earlier grid-based sub-watershed split with real
NHD/WBD HU12 polygons**, using `data_access.get_watershed_boundaries` from the
soil-watershed work. The grid split was always a placeholder so agency partners
work in HUC units, and a rectangular grid cannot be handed to anyone. That is the
substantive change in this version.

Four exposure questions:

1. Which structures sit in or near high-severity burn?
2. Which named communities carry the most exposure?
3. Which sub-watersheds drain burn scar toward municipal intakes?
4. Where does WUI structure (interface versus intermix) change the response?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

import daear_toolkit as dt
from daear_toolkit import data_access, indicators, viz, hydrology
from daear_toolkit import fire_access as fa
from daear_toolkit import fire_indicators as fi

REGION = dt.POUDRE_CAMERON_PEAK
BBOX = REGION.bbox

severity = xr.open_dataarray("../outputs/02_severity_class.nc")
recovery_ratio = xr.open_dataarray("../outputs/03_recovery_ratio.nc")
perimeters = fa.get_mtbs_perimeters(BBOX, year=2020, min_acres=1000)
terrain = data_access.get_terrain(BBOX)

## Structures

OSM building footprints via Overpass, reduced to centroids for point-in-class
counting.

**The completeness caveat governs every number in this module.** OSM coverage in
rural mountain Colorado is reasonable for permanent homes and poor for
outbuildings, seasonal cabins, and recent construction. Every structure count
below is a **lower bound** and must be labelled as one. Microsoft Building
Footprints is more complete and is the right swap for anything going to a
partner as it is a bulk download rather than an API, which is the only reason OSM
is the default here.

In [ ]:
buildings = fa.get_buildings(BBOX)
print(f"{len(buildings)} OSM building footprints in the AOI")
print("Types:", buildings["building"].value_counts().head(6).to_dict())

exposure = fi.exposure_by_zone(
    buildings, severity,
    zone_lookup={i: l for i, l in enumerate(fi.SEVERITY_LABELS)},
)
print("\nStructures by burn severity class (LOWER BOUND for OSM completeness):")
print(exposure)

fig, ax = plt.subplots(figsize=(8.5, 7))
viz.plot_raster(severity, title="Structures over burn severity", ax=ax, cmap=viz.SEVERITY_CMAP)
buildings.plot(ax=ax, color="black", markersize=2, alpha=0.6)
perimeters.boundary.plot(ax=ax, color="white", lw=1.0)
plt.tight_layout()
plt.savefig("../outputs/04_structures_severity.png", dpi=150)
plt.show()

## Distance-weighted exposure

A structure's risk is not only the severity of the pixel it stands on and it is
also what burned nearby. Ember cast and radiant heat mean adjacent
high-severity ground matters even when the structure itself sits on unburned
ground.

`decay_km` is a parameter, not a constant. Two values are run below, because a
single decay distance presented without sensitivity is a hidden assumption.

In [ ]:
intakes = fa.get_water_intakes()
print("Municipal intakes (approximate public locations: confirm with utilities):")
print(intakes[["name", "system", "approximate"]].to_string(index=False))

severity_continuous = severity.astype("float64") / 3.0   # 0-1 hazard surface

intake_exposure = pd.concat([
    fi.distance_weighted_exposure(intakes, severity_continuous, decay_km=km).assign(decay_km=km)
    for km in (2.0, 5.0)
], ignore_index=True)

print("\nDistance-weighted burn-severity exposure at municipal intakes:")
print(intake_exposure.pivot(index="name", columns="decay_km", values="weighted_hazard").round(3))
print("\nIf the ranking flips between decay values, the ranking is not robust and the")
print("finding is the exposure LEVEL, not the ordering.")

## Subwatersheds

WBD HU12 polygons replace the illustrative grid split.

HU12 units run roughly 40–160 km², which is the right grain: small enough that
the burn scar dominates some units and barely touches others, and that contrast
is what makes the ranking actionable. It is also the unit partners already have
in their own systems, so the output table joins to their data without
translation.

In [ ]:
huc12 = data_access.get_watershed_boundaries(BBOX, level=12, clip=True)
print(f"{len(huc12)} HU12 sub-watersheds intersect the AOI")

watershed_id, huc_lookup = data_access.rasterize_watersheds(huc12, template=severity)

# Flow routing, so "drains toward the intake" is a routed statement rather than
# a proximity assumption.
acc = hydrology.flow_accumulation(terrain["elevation"])
streams = hydrology.extract_streams(acc, threshold=500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
viz.plot_raster(watershed_id, title="HU12 sub-watersheds (real WBD polygons)", ax=axes[0], cmap="tab20")
huc12.boundary.plot(ax=axes[0], color="black", lw=0.5)
viz.plot_raster(severity, title="Severity with drainage network", ax=axes[1], cmap=viz.SEVERITY_CMAP)
viz.plot_raster(streams.where(streams > 0), ax=axes[1], cmap="Blues", add_colorbar=False)
for ax in axes:
    intakes.plot(ax=ax, color="cyan", markersize=60, marker="v", edgecolor="black", zorder=5)
plt.tight_layout()
plt.savefig("../outputs/04_subwatersheds.png", dpi=150)
plt.show()

In [ ]:
rows = []
for code, huc in huc_lookup.items():
    mask = watershed_id == code
    n = float(mask.sum())
    if n < 100:
        continue
    sev_in = severity.where(mask)
    struct = fi.exposure_by_zone(buildings, mask.where(mask))
    rows.append({
        "huc12": huc,
        "name": huc12.loc[huc12["huc12"] == huc, "name"].squeeze() if "name" in huc12 else "",
        "area_km2": round(n * 0.0004, 1),
        "pct_burned": round(float((sev_in > 0).mean()) * 100, 1),
        "pct_high_severity": round(float((sev_in == 3).mean()) * 100, 1),
        "mean_recovery_ratio": round(float(recovery_ratio.where(mask).mean()), 3),
        "structures": int(struct["n_assets"].sum()) if not struct.empty else 0,
    })

watersheds = pd.DataFrame(rows)
watersheds["priority_score"] = (
    watersheds["pct_high_severity"] / 100 * (1 - watersheds["mean_recovery_ratio"].clip(0, 1))
).round(3)
watersheds = watersheds.sort_values("priority_score", ascending=False)
watersheds.to_csv("../outputs/04_watershed_risk.csv", index=False)

print("Sub-watersheds ranked by post-fire priority (high severity x slow recovery):")
watersheds.head(10)

## WUI structure interface versus intermix

The Radeloff-style distinction, and it changes tactics rather than just
description:

- **Interface**: dense housing adjacent to wildland fuels. There is a defensible
  edge, and perimeter control is viable.
- **Intermix**: housing dispersed *within* continuous fuels. Defence is
  structure-by-structure, and evacuation timelines are longer.

Collapsing both into one "WUI" figure loses precisely the distinction an
emergency manager needs.

In [ ]:
from scipy.ndimage import uniform_filter

# Building density per km2 on the severity grid.
ydim, xdim = severity.dims
ys, xs = severity.coords[ydim].values, severity.coords[xdim].values
counts, _, _ = np.histogram2d(
    buildings.geometry.y.values, buildings.geometry.x.values,
    bins=[np.sort(ys), np.sort(xs)],
)
counts = np.flipud(counts) if ys[0] > ys[-1] else counts
padded = np.zeros(severity.shape)
padded[: counts.shape[0], : counts.shape[1]] = counts

cell_km2 = 0.0004   # 20 m pixel
density = xr.DataArray(uniform_filter(padded, size=25) / cell_km2, coords=severity.coords, dims=severity.dims)

veg_cover = (recovery_ratio.clip(0, 1) * 0.6 + 0.2)   # coarse proxy; LANDFIRE EVC is the better input
wui = fi.wui_classify(density, veg_cover)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
viz.plot_raster(density, title="Building density (structures/km2)", ax=axes[0], cmap="viridis")
viz.plot_raster(wui.where(wui > 0), title="WUI class (1=interface, 2=intermix)", ax=axes[1], cmap="Set1")
plt.tight_layout()
plt.savefig("../outputs/04_wui_classes.png", dpi=150)
plt.show()

for code, label in [(1, "interface"), (2, "intermix")]:
    area = float((wui == code).mean())
    burned = float(((wui == code) & (severity >= 2)).sum() / max((wui == code).sum(), 1))
    print(f"{label:<12} {area:5.1%} of AOI   |   {burned:5.1%} of it burned at moderate+ severity")

## Community summary table

Aggregated to named Census places as this is the frame a county emergency manager works
in.

**Places are a reporting frame, not a population denominator.** In Larimer County
a large share of WUI residents live in unincorporated areas that no place polygon
covers. For population figures, use block groups instead; for naming the
communities in a briefing, places are right.

In [ ]:
places = fa.get_places(BBOX)
print(f"{len(places)} Census places in the AOI")

rows = []
for _, place in places.iterrows():
    within = buildings[buildings.within(place.geometry)]
    if within.empty:
        continue
    exp = fi.exposure_by_zone(within, severity, zone_lookup={i: l for i, l in enumerate(fi.SEVERITY_LABELS)})
    high = int(exp.loc[exp["zone"] == 3, "n_assets"].sum()) if not exp.empty else 0
    mod_plus = int(exp.loc[exp["zone"] >= 2, "n_assets"].sum()) if not exp.empty else 0
    rows.append({
        "place": place.get("name") or place.get("basename"),
        "structures_osm": len(within),
        "in_high_severity": high,
        "in_moderate_plus": mod_plus,
        "pct_moderate_plus": round(100 * mod_plus / len(within), 1),
    })

community = pd.DataFrame(rows).sort_values("in_moderate_plus", ascending=False)
community.to_csv("../outputs/04_community_exposure.csv", index=False)
print("\nStructure counts are OSM-derived LOWER BOUNDS. Label them as such in any output.")
community

## Summary and what this dashboard is for

Structure exposure by severity class, distance-weighted exposure at municipal
intakes, sub-watersheds ranked on real HU12 units, WUI interface/intermix
classification, and a community summary table.

This closes the four-module arc: Module 1 established pre-fire conditions,
Module 2 measured what burned and tested whether conditions predicted it, Module
3 tracked recovery, and Module 4 translates all of it into exposure for people
and water supply.

**Read this as a screening tool, not a risk assessment.** The distinction is
practical:

- Structure counts are OSM lower bounds, not parcel records. A county assessor
  file would replace them and change the numbers.
- Intake locations are approximate public coordinates, flagged as such in the
  data. Confirm with the utilities before anything operational.
- The WUI vegetation input is a coarse proxy; LANDFIRE EVC is the correct layer
  and is a straightforward substitution.
- `priority_score` weights high severity against slow recovery equally, and that
  weighting is unvalidated. Run the weight-sensitivity Monte Carlo from
  `climate_indicators` on it before presenting the ranking as a ranking.

**What would make it operational:** county parcel data, confirmed infrastructure
locations, and a conversation with Larimer County Office of Emergency Management
about which questions they actually need answered. The analysis is the easy part;
that conversation determines whether any of it gets used.